- 从 (B,T,d) 的处理过程看最清楚：
    - `(B,T,d) --reshape--> (B·T, d) --@W1--> (B·T, 4d) --act--> --@W2--> (B·T, d) --reshape--> (B,T,d)`
    - T 被折进了 batch 维，token 之间在这一步是完全等价、可任意重排的。写成 einsum 对比：
    ```
    FFN : y[b,t,j] = Σ_i  x[b,t,i] · W[i,j]      # 求和指标 i 是 hidden 维
    Attn: y[b,t,j] = Σ_s  A[b,t,s] · v[b,s,j]    # 求和指标 s 是另一个位置
    ```
    - 被求和掉（contract 掉）的指标里，有没有"另一个位置"。FFN 里 t 从头到尾是自由指标，只是跟着走，从来没被求和掉;attention 里 s 被求和掉了。这就是 token interaction 的定义。
- 训练和推理的角度
    - 自回归 decode 的时候，FFN 的输入 shape 就是 (B, 1, d)。 attention 必须读取整个 KV cache，而 FFN 只吃当前这一个 token，前面的一个都不要。
    - FN 之所以不需要任何 cache，正是因为它没有跨位置依赖。如果 FFN 真需要序列全局信息，每生成一个 token 都得把整条序列的 FFN 重算一遍。
- 共享与全局
    - FFN 本身不负责 token 之间的信息传递；跨 token 依赖通常来自注意力（Attention）。
    - token 之间共享的是权重不是信息。同一个函数 $f$ 分别作用在 T 个向量上，$f$ 是共享的，但 $f(x_1)$ 的结果里没有 $x_2$ 的任何成分。这跟 CNN 卷积核在空间上共享是一回事——FFN 其实就是 kernel_size=1 的卷积（早期 fairseq 就直接用 nn.Conv1d(d, 4d, 1) 写 FFN，和 Linear 数值完全等价），kernel size = 1 意味着感受野只有当前这一格。权重共享是为了让参数量与 T 无关、能处理变长序列，不是为了让信息跨位置流动。
    - FFN 的权重确实"编码了全局知识"，但那是训练数据里的知识、固化在参数里、与当前这条序列无关；
    - attention 取的是当前这条序列里其他位置的信息、依赖输入、每次都不同。前者是各位置独立地从同一份权重里读，后者是从别的位置读。这两个"全局"不是一回事。
- X -> Q/K/V 的过程也是 linear projection

### math understanding

脚本中批大小 $B=1$，因此省略 batch 维。令

$$X=\begin{bmatrix}x_0^\top\\x_1^\top\\\vdots\\x_{T-1}^\top\end{bmatrix}\in\mathbb{R}^{T\times d},
\qquad Y=f(X)=
\begin{bmatrix}
y_0^\top\\
y_1^\top\\
\vdots\\
y_{T-1}^\top
\end{bmatrix}
\in\mathbb{R}^{T\times d},
$$

其中 $T=6$、$d=8$，位置编号采用与代码一致的零基编号 $t,s\in\{0,\ldots,T-1\}$。记 $f_t(X)=y_t\in\mathbb{R}^d$ 为网络在位置 $t$ 的输出。这里比较三个映射：$f_{\mathrm{FFN}}, f_{\mathrm{bi}}, f_{\mathrm{causal}}.$

- 逐 token FFN 为 $f_{\mathrm{FFN},t}(X)=W_2\,\operatorname{GELU}(W_1x_t+b_1)+b_2.$
    - 因此 $y_t$ 只依赖 $x_t$，不同位置之间共享同一个 FFN 参数，但不交换信息。
- 对第 $h$ 个注意力头
    - $q_t^{(h)}=W_Q^{(h)}x_t,k_s^{(h)}=W_K^{(h)}x_s,v_s^{(h)}=W_V^{(h)}x_s,$
    - $\alpha_{ts}^{(h)}=\frac{
\exp\left(\frac{\langle q_t^{(h)},k_s^{(h)}\rangle}{\sqrt{d_h}}+m_{ts}\right)}{
\sum_{u=0}^{T-1}\exp\left(\frac{\langle q_t^{(h)},k_u^{(h)}\rangle}{\sqrt{d_h}}+m_{tu}\right)
},$
    - $y_t=W_O\operatorname{Concat}_{h}\left(\sum_{s=0}^{T-1}\alpha_{ts}^{(h)}v_s^{(h)}\right)$
    - 双向注意力中 $m_{ts}=0$。因果注意力中$m_{ts}=\begin{cases}0,&s\le t,\\-\infty,&s>t.\end{cases}$

In [10]:
import torch
import torch.nn as nn

torch.manual_seed(0)
B, T, d = 1, 6, 8
PERTURB_AT = 2                       # 第 1 项扰动的位置
REPLACE_S, PROBE_T = 1, 4            # 第 4 项：替换位置 s，观察位置 t

x = torch.randn(B, T, d)
ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
mha = nn.MultiheadAttention(d, num_heads=2, batch_first=True)
causal = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)

attn_bidir = lambda z: mha(z, z, z, need_weights=False)[0]
attn_causal = lambda z: mha(z, z, z, attn_mask=causal, need_weights=False)[0]
PATHS = [("FFN        ", ffn), ("Attn-bidir ", attn_bidir), ("Attn-causal", attn_causal)]

#### probe1：单点扰动

In [8]:
# --- 1. 单点扰动（图 (c) 的读数，只取 FFN 与 causal 两行）-------------------
x2 = x.clone()
x2[:, PERTURB_AT, :] += 1.0
print(f"[1] 只把 x[:, {PERTURB_AT}, :] 加 1.0，实测 |dy_t|_1")
for name, fn in PATHS:
    delta = (fn(x2) - fn(x)).abs().sum(-1).squeeze()
    print(f"    {name} {[f'{v:.3f}' for v in delta.tolist()]}")

[1] 只把 x[:, 2, :] 加 1.0，实测 |dy_t|_1
    FFN         ['0.000', '0.000', '1.705', '0.000', '0.000', '0.000']
    Attn-bidir  ['0.299', '0.276', '1.335', '0.287', '0.495', '0.314']
    Attn-causal ['0.000', '0.000', '1.634', '0.449', '0.582', '0.314']


令扰动位置为 $s_0=2$，并将该 token 的所有特征同时增加 $1$：
$$
X^{\mathrm{add}}
=X+e_{s_0}\mathbf{1}_d^\top,$$

其中 $e_{s_0}\in\mathbb{R}^T$ 是第 $s_0$ 个标准基向量，$\mathbf{1}_d\in\mathbb{R}^d$ 是全一向量。这正对应
x2[:, 2, :] += 1.0
位置 $t$ 的输出变化量定义为
$$
\Delta_t^{\mathrm{add}}
=\left\|f_t(X^{\mathrm{add}})-f_t(X)
\right\|_1
=\sum_{i=1}^{d}
\left|
f_t(X^{\mathrm{add}})_i-f_t(X)_i
\right|.
$$
这是有限幅度扰动（finite perturbation），而不是无穷小导数。
理论上的稀疏模式为：
$$
\begin{array}{c|c}
\text{映射}&\Delta_t^{\mathrm{add}}\text{ 可能非零的位置}\\
\hline
\text{FFN}&t=2\\
\text{双向注意力}&t=0,\ldots,T-1\\
\text{因果注意力}&t\ge 2
\end{array}
$$因果注意力中 $t<2$ 的输出不能读取未来位置 $2$，所以这些位置的变化量应为零。

In [11]:
# --- 2. 雅可比块范数（图 (c) 每格的填充深度）--------------------------------
for name, fn in PATHS[::2]:
    J = torch.autograd.functional.jacobian(lambda z: fn(z).squeeze(0), x)  # (T,d,B,T,d)
    M = J.squeeze(2).permute(0, 2, 1, 3).flatten(2).norm(dim=-1)           # (t,s)
    print(f"\n[2] {name} ||dy_t/dx_s||_F")
    for row in M.tolist():
        print("    " + " ".join(f"{v:5.2f}" for v in row))


[2] FFN         ||dy_t/dx_s||_F
     0.64  0.00  0.00  0.00  0.00  0.00
     0.00  0.52  0.00  0.00  0.00  0.00
     0.00  0.00  0.58  0.00  0.00  0.00
     0.00  0.00  0.00  0.60  0.00  0.00
     0.00  0.00  0.00  0.00  0.54  0.00
     0.00  0.00  0.00  0.00  0.00  0.64

[2] Attn-causal ||dy_t/dx_s||_F
     1.00  0.00  0.00  0.00  0.00  0.00
     0.49  0.54  0.00  0.00  0.00  0.00
     0.19  0.60  0.67  0.00  0.00  0.00
     0.31  0.25  0.21  0.40  0.00  0.00
     0.21  0.25  0.16  0.22  0.32  0.00
     0.23  0.09  0.16  0.16  0.17  0.48


#### probe2：雅可比块范数

把完整雅可比矩阵（Jacobian）按输入、输出 token 位置划分为 $T\times T$ 个 $d\times d$ 块：
$$
J_{ts}(X)
=\frac{\partial y_t}{\partial x_s}
=\frac{\partial f_t(X)}{\partial x_s}
\in\mathbb{R}^{d\times d}.
$$其中元素
$$
[J_{ts}]_{ij}
=\frac{\partial y_{t,i}}{\partial x_{s,j}}
$$表示输入位置 $s$ 的第 $j$ 个特征对输出位置 $t$ 的第 $i$ 个特征的一阶影响。

每个块用 Frobenius 范数压缩为一个标量：
$$
M_{ts}
=\|J_{ts}\|_F
=\sqrt{
\sum_{i=1}^{d}
\sum_{j=1}^{d}
\left(
\frac{\partial y_{t,i}}{\partial x_{s,j}}
\right)^2
}.
$$
于是得到矩阵
$$
M\in\mathbb{R}^{T\times T},
\qquad
M_{ts}=\left\|\frac{\partial y_t}{\partial x_s}\right\|_F.
$$代码中 torch.autograd.functional.jacobian 首先得到索引结构
$$
J[t,i,0,s,j]
=
\frac{\partial y_{t,i}}{\partial x_{s,j}},
$$随后去掉大小为 $1$ 的 batch 维，并将每个 $(d,d)$ 块展平后计算二范数；这正是上述 Frobenius 范数。

三类结构分别应为：

$$
M^{\mathrm{FFN}}
=
\begin{bmatrix}
* & 0 & \cdots & 0\\
0 & * & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots\\
0 & 0 & \cdots & *
\end{bmatrix},
$$

$$
M^{\mathrm{bi}}
\approx
\begin{bmatrix}
* & * & \cdots & *\\
* & * & \cdots & *\\
\vdots & \vdots & \ddots & \vdots\\
* & * & \cdots & *
\end{bmatrix},
$$

$$
M^{\mathrm{causal}}
\approx
\begin{bmatrix}
* & 0 & 0 & \cdots\\
* & * & 0 & \cdots\\
* & * & * & \cdots\\
\vdots & \vdots & \vdots & \ddots
\end{bmatrix}.
$$

#### probe3: 置换等变性

In [12]:
g = torch.Generator().manual_seed(1)          # 独立发生器，不扰动上面的复现性
perm = torch.randperm(T, generator=g)
inv = perm.argsort()
print(f"\n[3] 置换等变  perm = {perm.tolist()}")
for name, fn in PATHS:
    err = (fn(x[:, perm])[:, inv] - fn(x)).abs().max().item()
    print(f"    {name} max|P^-1 f(Px) - f(x)| = {err:.2e}   {'等变' if err < 1e-5 else '不等变'}")


[3] 置换等变  perm = [1, 5, 2, 0, 3, 4]
    FFN         max|P^-1 f(Px) - f(x)| = 7.45e-08   等变
    Attn-bidir  max|P^-1 f(Px) - f(x)| = 8.94e-08   等变
    Attn-causal max|P^-1 f(Px) - f(x)| = 2.06e-01   不等变


逐 token FFN 满足置换等变性，因为
$$
f_{\mathrm{FFN},t}(P_\pi X)
=
g(x_{\pi(t)})
=
(P_\pi f_{\mathrm{FFN}}(X))_t.
$$
不含位置编码的双向 self-attention 也满足置换等变性。排列输入只会同步排列 query、key、value 以及最终输出，并不会改变 token 之间的成对内容关系。